In [ ]:
%%capture
!pip install --user "ibm-watsonx-ai==0.2.6"
!pip install --user "langchain==0.1.16" 
!pip install --user "langchain-ibm==0.1.4"
!pip install --user "langchain-experimental==0.0.57"
!pip install --user "matplotlib==3.8.4"
!pip install --user "seaborn==0.13.2"

In [ ]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Load sample dataset

df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ZNoKMJ9rssJn-QbJ49kOzA/student-mat.csv"
)

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
## LOAD LLM
# Create a dictionary to store credential information
credentials = {
    "url"    : "https://us-south.ml.cloud.ibm.com"
}

# Indicate the model you would like to initialize. In this case, Llama 4 Maverick.
model_id    ='meta-llama/llama-4-maverick-17b-128e-instruct-fp8'

# Initialize some watsonx.ai model parameters
params = {
        GenParams.MAX_NEW_TOKENS: 256, # The maximum number of tokens that the model can generate in a single run.
        GenParams.TEMPERATURE: 0,   # A parameter that controls the randomness of the token generation. A lower value makes the generation more deterministic, while a higher value introduces more randomness.
    }
project_id  = "skills-network" # <--- NOTE: specify "skills-network" as your project_id
space_id    = None
verify      = False

# Launch a watsonx.ai model
model = Model(
    model_id=model_id, 
    credentials=credentials, 
    params=params, 
    project_id=project_id, 
    space_id=space_id, 
    verify=verify
)

# Integrate the watsonx.ai model with the langchain framework
llm = WatsonxLLM(model = model)

agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=False,
    return_intermediate_steps=True,  # set return_intermediate_steps=True so that model could return code that it comes up with to generate the chart
    handle_parsing_errors=True,
    prefix="You are a pandas agent. Always respond using Action/Action Input/Final Answer format. Never output raw data directly."
)

In [ ]:
# IGNORE IF YOU ARE NOT RUNNING LOCALLY

credentials = {
    "url"    : "https://us-south.ml.cloud.ibm.com",
    "api_key": "your api key here"
}

model = Model(
    model_id=model_id, 
    credentials=credentials, 
    params=params, 
    project_id=project_id, 
    space_id=space_id, 
    verify=verify
)

llm = WatsonxLLM(model = model)

In [ ]:
## INTERACTING WITH THE DATA
response = agent.invoke("how many rows of data are in this file?")

In [ ]:
response['output']

In [ ]:
len(df)

In [ ]:
response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n')

In [ ]:
response = agent.invoke("Give me all the data where student's age is over 18 years old.")

In [ ]:
print(response)

In [ ]:
response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n')

In [ ]:
response = agent.invoke("Generate a bar chart to plot the gender count.")

In [ ]:
print(response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n'))

In [ ]:
response = agent.invoke("Generate a pie chart to display average value of Walc for each Gender.")

In [ ]:
print(response['intermediate_steps'][-1][0].tool_input.replace('; ', '\n'))

In [ ]:
response = agent.invoke("Create box plots to analyze the relationship between 'freetime' (amount of free time) and 'G3' (final grade) across different levels of free time.")